# Aleph Direct-A 10M -> 100M (Staged 10B) Continuation Launcher — Google Cloud TPU v5e-1

Turnkey capped Colab launcher for the frozen **999,352,320-parameter** Aleph Direct-A continuation runner on **1x Google Cloud TPU `v5e-1`** (`3.26 CU/hr`, **`3,275 pos/s`**, **`~24.9 CU` for all 90M tokens** — 45% cheaper than A100's `45.1 CU`).

- **Runtime:** Select **Runtime -> Change runtime type -> `v5e-1 TPU`**, then click **Run all**.
- **Never embeds secrets or tokens.** Uses Colab's native Google Drive mount.
- **Immutable prior checkpoint:** reads `MyDrive/aleph-v2-run` read-only and never modifies `group_039_7fbdb1f91cf8`.
- **Fail-closed qualification + Drive restart parity:** automatically qualifies groups 40–41 + Drive restore parity before scaling in resumable 25-group chunks to absolute step `383` (`100,000,000` positions), then scores 10M vs 100M on the disjoint `dev_panel`.

In [ ]:
# === OPERATOR CONFIGURATION ===
REPO_URL = "https://github.com/MoralityLabAI/DataGoblin.git"
REPO_BRANCH = "feat/aleph-10b-continuation-prep-20261128"

DEVICE = "tpu"  # "tpu" (Colab TPU v5e-1 3.26 CU/hr, 3,275 pos/s) or "cuda" (A100)
RUN_NAME = "stage_100m_tpu_v5e1"
CHUNK_GROUPS = 25  # Sync verified checkpoint to Drive every 25 groups (~33 min/chunk)
TARGET_ABS_STEP = 383  # 39 prior + 344 Stage-1 groups = 100,000,000 cumulative positions
SESSION_WALL_SECONDS = 32400  # 9-hour hard cap (~29.3 CU max spend)
RUN_DEV_EVAL_AT_END = True  # Score 10M (step 39) vs 100M (step 383) on disjoint dev_panel
AUTO_DISCONNECT_ON_FINISH = True  # Release TPU automatically when finished or wall-capped


In [ ]:
# 1. Mount Google Drive and clone pinned DataGoblin trainer
import shutil
import subprocess
import sys
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

repo_dir = Path("/content/DataGoblin")
if not repo_dir.exists():
    subprocess.run(["git", "clone", "--branch", REPO_BRANCH, "--depth", "1", REPO_URL, str(repo_dir)], check=True)
else:
    subprocess.run(["git", "-C", str(repo_dir), "fetch", "origin", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(repo_dir), "checkout", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(repo_dir), "reset", "--hard", f"origin/{REPO_BRANCH}"], check=True)

for label, p in [("local_nvme", "/content"), ("google_drive", "/content/drive/MyDrive")]:
    du = shutil.disk_usage(p)
    print(f"{label}: free={du.free / 1e9:.2f} GB / total={du.total / 1e9:.2f} GB")


In [ ]:
# 2. Run qualification (groups 40-41 + Drive restart parity) -> full Stage 1 (step 383) -> dev eval
driver_py = repo_dir / "train/aleph_continuation10b/stage1_tpu_driver.py"
eval_py = repo_dir / "train/aleph_continuation10b/eval_dev_panel.py"

try:
    subprocess.run(
        [
            sys.executable,
            str(driver_py),
            "--device",
            DEVICE,
            "--run-name",
            RUN_NAME,
            "--chunk-groups",
            str(CHUNK_GROUPS),
            "--target-abs-step",
            str(TARGET_ABS_STEP),
            "--session-wall-seconds",
            str(SESSION_WALL_SECONDS),
        ],
        check=True,
    )
    if RUN_DEV_EVAL_AT_END:
        subprocess.run(
            [sys.executable, str(eval_py), "--device", DEVICE, "--run-name", RUN_NAME],
            check=True,
        )
finally:
    if AUTO_DISCONNECT_ON_FINISH:
        from google.colab import runtime

        runtime.unassign()
